> **REQUISITO:** este notebook funciona con el perfil **basico** de Docker.
> ```bash
> docker-compose --profile basico up -d
> ```
> Usa el **metastore embebido** de Spark (Apache Derby), por lo que NO
> necesitas levantar el servicio externo de Hive. Las tablas quedan
> registradas en `/home/jovyan/work/metastore_db` y los datos en el
> warehouse compartido `/warehouse` (ambos persisten entre sesiones).

# EA2 - Actividad 2.3b: Hive Metastore con Spark

## Objetivos
- Conectar Spark a Hive Metastore
- Crear tablas persistentes y externas
- Entender la diferencia entre tablas gestionadas y externas
- Aplicar particionamiento de datos
- Ejecutar consultas SQL sobre tablas Hive

## Conceptos Clave

### Apache Hive como Data Warehouse

Apache Hive es un sistema de data warehouse que permite consultar grandes
volumenes de datos usando SQL. El componente central es el **Metastore**, que
almacena los metadatos de las tablas (schema, ubicacion, formato, particiones).

### Hive Metastore

El Metastore es un catalogo que almacena:
- **Bases de datos** y sus ubicaciones
- **Esquemas de tablas** (columnas, tipos de datos)
- **Ubicacion de los datos** en el sistema de archivos
- **Informacion de particiones**

> **En este entorno** usamos el metastore **embebido** de Spark (Apache Derby):
> es local al notebook y no requiere un servicio aparte. En produccion o en la
> nube, el metastore es un servicio **remoto** (Spark se conecta via el protocolo
> **Thrift**) o **gestionado** (AWS Glue Data Catalog, Dataproc Metastore,
> Azure Synapse/Purview). El codigo SQL y los conceptos son identicos.

### Tablas Gestionadas vs Externas

| Caracteristica | Tabla Gestionada (Managed) | Tabla Externa (External) |
|----------------|---------------------------|-------------------------|
| Datos | Spark gestiona los datos | Los datos existen externamente |
| DROP TABLE | Elimina datos + metadata | Solo elimina metadata |
| Uso tipico | Datos procesados internamente | Datos compartidos entre herramientas |
| Comando | `saveAsTable()` | `CREATE EXTERNAL TABLE` |

### Apache Parquet

Parquet es un formato **columnar** optimizado para Big Data:
- Compresion eficiente por columna
- Lectura selectiva de columnas (projection pushdown)
- Schema embebido en el archivo
- Formato estandar en el ecosistema Hadoop/Spark

### Particionamiento

El particionamiento divide los datos en subdirectorios basados en el valor de
una o mas columnas:
```
vuelos_por_mes/
  MONTH=1/
    part-00000.parquet
  MONTH=2/
    part-00000.parquet
  ...
```
Esto acelera las consultas que filtran por la columna de particion, ya que
Spark solo lee los directorios relevantes (**partition pruning**).

## Setup

In [1]:
from pyspark.sql import SparkSession

# Spark usa su metastore EMBEBIDO (Apache Derby). No requiere el servicio
# externo de Hive: el cliente Hive de Spark 4.1 no es compatible con el
# metastore Hive 4.0, asi que el embebido es la opcion fiable en este entorno.
spark = (
    SparkSession.builder
    .appName("hive_metastore")
    .master("local[*]")
    # Warehouse: donde se guardan los datos de las tablas (volumen compartido)
    .config("spark.sql.warehouse.dir", "/warehouse")
    # Metastore embebido persistente (Derby) en el volumen de trabajo
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        "jdbc:derby:;databaseName=/home/jovyan/work/metastore_db;create=true",
    )
    .enableHiveSupport()
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("Hive support habilitado (metastore embebido Derby)")
# Nota: la primera celda puede tardar ~1 min mientras se inicializa el metastore.

Spark version: 4.1.2
Hive support habilitado (metastore embebido Derby)


## 1. Verificar Conexion al Metastore

Lo primero es verificar que Spark puede comunicarse con el Hive Metastore. Si este comando falla, asegurate de que el servicio `hive-metastore` este corriendo.

In [2]:
# Verificar conexion listando las bases de datos disponibles
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



## 2. Crear Base de Datos

Las bases de datos en Hive son contenedores logicos para organizar tablas. Crearemos una base de datos llamada `bigdata` para nuestro proyecto.

In [3]:
# Crear base de datos (IF NOT EXISTS evita error si ya existe)
spark.sql("CREATE DATABASE IF NOT EXISTS bigdata")

# Verificar que se creo
spark.sql("SHOW DATABASES").show()

# Usar la base de datos
spark.sql("USE bigdata")
print("Base de datos 'bigdata' seleccionada")

+---------+
|namespace|
+---------+
|  bigdata|
|  default|
+---------+

Base de datos 'bigdata' seleccionada


## 3. Crear Tabla Gestionada desde DataFrame

Una **tabla gestionada** (managed table) es aquella donde Spark controla tanto los datos como los metadatos. Al usar `saveAsTable()`, Spark escribe los datos en el warehouse de Hive y registra la tabla en el Metastore.

In [4]:
# Leer datos de vuelos
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True, nullValue="NA")

print(f"Registros cargados: {df.count()}")
df.printSchema()

Registros cargados: 162049
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: integer (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- arr_time: integer (nullable = true)
 |-- arr_delay: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)



In [5]:
# Guardar como tabla gestionada en Hive
df.write.mode("overwrite").saveAsTable("bigdata.vuelos")

# Verificar que la tabla existe
spark.sql("SHOW TABLES IN bigdata").show()

# Consultar la tabla
spark.sql("SELECT COUNT(*) FROM bigdata.vuelos").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  bigdata|   vuelos|      false|
+---------+---------+-----------+



+--------+
|count(1)|
+--------+
|  162049|
+--------+



In [6]:
# Ver informacion detallada de la tabla
spark.sql("DESCRIBE EXTENDED bigdata.vuelos").show(truncate=False)

+----------------------------+-------------+-------+
|col_name                    |data_type    |comment|
+----------------------------+-------------+-------+
|year                        |int          |NULL   |
|month                       |int          |NULL   |
|day                         |int          |NULL   |
|dep_time                    |int          |NULL   |
|dep_delay                   |int          |NULL   |
|arr_time                    |int          |NULL   |
|arr_delay                   |int          |NULL   |
|carrier                     |string       |NULL   |
|tailnum                     |string       |NULL   |
|flight                      |int          |NULL   |
|origin                      |string       |NULL   |
|dest                        |string       |NULL   |
|air_time                    |int          |NULL   |
|distance                    |int          |NULL   |
|hour                        |int          |NULL   |
|minute                      |int          |NU

## 4. Crear Tabla Externa apuntando a Parquet

Una **tabla externa** no gestiona los datos; solo apunta a una ubicacion existente. Al eliminar la tabla, los datos permanecen intactos.

Primero guardamos datos en formato Parquet, y luego creamos una tabla externa que apunta a esa ubicacion.

In [7]:
# Guardar un subconjunto de datos como Parquet en el warehouse compartido
df.select("year", "month", "day", "carrier", "origin", "dest", "dep_delay", "arr_delay", "distance") \
    .write.mode("overwrite") \
    .parquet("/warehouse/vuelos_parquet")

print("Datos guardados en formato Parquet en /warehouse/vuelos_parquet")

Datos guardados en formato Parquet en /warehouse/vuelos_parquet


In [8]:
# Crear tabla externa apuntando a los archivos Parquet
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS bigdata.vuelos_ext (
        year INT,
        month INT,
        day INT,
        carrier STRING,
        origin STRING,
        dest STRING,
        dep_delay INT,
        arr_delay INT,
        distance INT
    )
    STORED AS PARQUET
    LOCATION '/warehouse/vuelos_parquet'
""")

# Verificar
spark.sql("SELECT COUNT(*) as total FROM bigdata.vuelos_ext").show()
spark.sql("SELECT * FROM bigdata.vuelos_ext LIMIT 5").show()

+------+
| total|
+------+
|162049|
+------+



+----+-----+---+-------+------+----+---------+---------+--------+
|year|month|day|carrier|origin|dest|dep_delay|arr_delay|distance|
+----+-----+---+-------+------+----+---------+---------+--------+
|2014|    1|  1|     AS|   PDX| ANC|       96|       70|    1542|
|2014|    1|  1|     US|   SEA| CLT|       -6|      -23|    2279|
|2014|    1|  1|     UA|   PDX| IAH|       13|       -4|    1825|
|2014|    1|  1|     US|   PDX| CLT|       -2|      -23|    2282|
|2014|    1|  1|     AS|   SEA| ANC|       44|       43|    1448|
+----+-----+---+-------+------+----+---------+---------+--------+



## 5. Consultas SQL sobre Tablas Hive

Una vez que las tablas estan registradas en el Metastore, podemos ejecutar consultas SQL directamente sobre ellas, como si fueran tablas en una base de datos tradicional.

In [9]:
# Listar todas las tablas disponibles
spark.sql("SHOW TABLES IN bigdata").show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  bigdata|    vuelos|      false|
|  bigdata|vuelos_ext|      false|
+---------+----------+-----------+



In [10]:
# Consulta: conteo de vuelos por aerolinea
spark.sql("""
    SELECT carrier, COUNT(*) as total_vuelos
    FROM bigdata.vuelos
    GROUP BY carrier
    ORDER BY total_vuelos DESC
""").show()

+-------+------------+
|carrier|total_vuelos|
+-------+------------+
|     AS|       62460|
|     WN|       23355|
|     OO|       18710|
|     DL|       16716|
|     UA|       16671|
|     AA|        7586|
|     US|        5946|
|     B6|        3540|
|     VX|        3272|
|     F9|        2698|
|     HA|        1095|
+-------+------------+



In [11]:
# Consulta: promedio de retraso por mes
# dep_delay y arr_delay son IntegerType (gracias a nullValue="NA" en la lectura)
spark.sql("""
    SELECT month, 
           ROUND(AVG(dep_delay), 2) as avg_retraso_salida,
           ROUND(AVG(arr_delay), 2) as avg_retraso_llegada
    FROM bigdata.vuelos
    WHERE dep_delay IS NOT NULL AND arr_delay IS NOT NULL
    GROUP BY month
    ORDER BY month
""").show()

+-----+------------------+-------------------+
|month|avg_retraso_salida|avg_retraso_llegada|
+-----+------------------+-------------------+
|    1|              7.25|               2.12|
|    2|              8.46|               2.83|
|    3|               5.1|                0.6|
|    4|              3.62|              -0.56|
|    5|              4.01|                1.0|
|    6|              7.52|               3.74|
|    7|               7.6|               4.67|
|    8|              6.01|               3.67|
|    9|              3.86|               0.77|
|   10|              3.92|               0.22|
|   11|              6.02|              -0.19|
|   12|             10.29|               7.18|
+-----+------------------+-------------------+



In [12]:
# Consulta: top 10 rutas mas frecuentes
spark.sql("""
    SELECT origin, dest, 
           COUNT(*) as total_vuelos
    FROM bigdata.vuelos_ext
    GROUP BY origin, dest
    ORDER BY total_vuelos DESC
    LIMIT 10
""").show()

+------+----+------------+
|origin|dest|total_vuelos|
+------+----+------------+
|   SEA| SFO|        7630|
|   SEA| LAX|        7455|
|   SEA| ANC|        6149|
|   SEA| LAS|        5732|
|   SEA| DEN|        5578|
|   PDX| SFO|        5179|
|   SEA| PHX|        5090|
|   SEA| ORD|        4520|
|   SEA| SJC|        3950|
|   PDX| DEN|        3940|
+------+----+------------+



## 6. Particionamiento de Tablas

El particionamiento organiza los datos en subdirectorios segun el valor de una columna. Esto mejora drasticamente el rendimiento de consultas que filtran por esa columna, ya que Spark solo necesita leer las particiones relevantes (**partition pruning**).

In [13]:
# Crear tabla particionada por month
df.write.mode("overwrite").partitionBy("month").saveAsTable("bigdata.vuelos_por_mes")

print("Tabla particionada creada exitosamente")

Tabla particionada creada exitosamente


In [14]:
# Ver las particiones disponibles
spark.sql("SHOW PARTITIONS bigdata.vuelos_por_mes").show()

+---------+
|partition|
+---------+
|  month=1|
| month=10|
| month=11|
| month=12|
|  month=2|
|  month=3|
|  month=4|
|  month=5|
|  month=6|
|  month=7|
|  month=8|
|  month=9|
+---------+



In [15]:
# Consulta optimizada: solo lee la particion month=1
# Gracias al partition pruning, Spark no lee los datos de otros meses
spark.sql("""
    SELECT carrier, COUNT(*) as vuelos_enero
    FROM bigdata.vuelos_por_mes
    WHERE month = 1
    GROUP BY carrier
    ORDER BY vuelos_enero DESC
""").show()

+-------+------------+
|carrier|vuelos_enero|
+-------+------------+
|     AS|        4686|
|     OO|        1764|
|     WN|        1611|
|     UA|        1177|
|     DL|        1077|
|     AA|         581|
|     US|         426|
|     VX|         295|
|     B6|         229|
|     F9|         216|
|     HA|          93|
+-------+------------+



In [16]:
# Comparar: descripcion de tabla particionada vs no particionada
print("=== Tabla NO particionada ===")
spark.sql("DESCRIBE EXTENDED bigdata.vuelos").show(truncate=False)

print("\n=== Tabla particionada ===")
spark.sql("DESCRIBE EXTENDED bigdata.vuelos_por_mes").show(truncate=False)

=== Tabla NO particionada ===


+----------------------------+-------------+-------+
|col_name                    |data_type    |comment|
+----------------------------+-------------+-------+
|year                        |int          |NULL   |
|month                       |int          |NULL   |
|day                         |int          |NULL   |
|dep_time                    |int          |NULL   |
|dep_delay                   |int          |NULL   |
|arr_time                    |int          |NULL   |
|arr_delay                   |int          |NULL   |
|carrier                     |string       |NULL   |
|tailnum                     |string       |NULL   |
|flight                      |int          |NULL   |
|origin                      |string       |NULL   |
|dest                        |string       |NULL   |
|air_time                    |int          |NULL   |
|distance                    |int          |NULL   |
|hour                        |int          |NULL   |
|minute                      |int          |NU

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [17]:
# =============================================================
# EJERCICIO 1: Crear tabla particionada por carrier
# =============================================================
# TODO: Crea una tabla llamada "bigdata.vuelos_particionada"
#   particionada por la columna carrier.
#
# Pasos:
#   1. Usar df.write.mode("overwrite").partitionBy("carrier")
#      .saveAsTable("bigdata.vuelos_particionada")
#   2. Verificar las particiones con SHOW PARTITIONS
#   3. Ejecutar una consulta que filtre por una aerolinea
#      especifica y contar los vuelos
#
# Pista: Recuerda que el particionamiento crea un subdirectorio
#   por cada valor unico de la columna carrier.

# Escribe tu codigo aqui:


In [18]:
# =============================================================
# EJERCICIO 2: Consultas SQL sobre tablas Hive
# =============================================================
# TODO: Ejecuta 3 queries SQL diferentes sobre las tablas Hive:
#
# Query 1: Conteo de vuelos por mes, ordenado por mes
#   - Tabla: bigdata.vuelos_por_mes
#   - Columnas: month, total_vuelos
#
# Query 2: Promedio de retraso de salida por aerolinea
#   - Tabla: bigdata.vuelos_particionada
#   - Columnas: carrier, avg_retraso
#   - Nota: dep_delay es IntegerType (gracias a nullValue="NA")
#   - Filtrar: solo donde dep_delay IS NOT NULL
#   - Ordenar: por avg_retraso descendente
#
# Query 3: Top 5 rutas con mas vuelos
#   - Tabla: bigdata.vuelos
#   - Columnas: origin, dest, total
#   - Ordenar: por total descendente, LIMIT 5

# Escribe tu codigo aqui:

# Query 1:


# Query 2:


# Query 3:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Disenar un schema de data warehouse para datos de ventas.

In [19]:
# =============================================================
# DESAFIO: Disenar un Data Warehouse en Hive
# =============================================================
# TODO: Disenar e implementar un schema de data warehouse
#   para los datos de ventas (sales.csv + stores.csv).
#
# Requisitos:
#   1. Crear una tabla de HECHOS (fact table): fact_ventas
#      - Contiene: Store, Dept, Date, Weekly_Sales, IsHoliday
#      - Particionada por algun criterio (ej: IsHoliday o Store)
#
#   2. Crear una tabla de DIMENSION: dim_tiendas
#      - Contiene: Store, Type, Size
#      - Datos de stores.csv
#
# Pasos sugeridos:
#   a) Leer sales.csv y stores.csv
#   b) Crear base de datos: bigdata_warehouse
#   c) Guardar dim_tiendas como tabla gestionada
#   d) Guardar fact_ventas particionada
#   e) Ejecutar query que combine ambas tablas con JOIN
#      Ejemplo: ventas totales por tipo de tienda
#
# Pista:
#   df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
#   df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Hive Metastore:** Servicio centralizado que almacena metadatos de tablas, schemas y particiones
2. **Conexion Spark-Hive:** Usar `enableHiveSupport()` y configurar `hive.metastore.uris`
3. **Bases de datos:** `CREATE DATABASE` para organizar tablas logicamente
4. **Tablas gestionadas:** `saveAsTable()` - Spark controla datos y metadata
5. **Tablas externas:** `CREATE EXTERNAL TABLE` - Solo metadata, datos externos
6. **Formato Parquet:** Formato columnar eficiente para Big Data
7. **Particionamiento:** `partitionBy()` para organizar datos en subdirectorios y optimizar consultas
8. **Consultas SQL:** Queries directas sobre tablas persistentes en Hive

In [20]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")

SparkSession detenida correctamente.
